In [ ]:
from crewai import LLM, Crew, Agent, Task
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import SentenceTransformerEmbeddings  # Use the wrapper


#from langchain.agents import tool
from crewai.tools import tool

In [ ]:
from com.example.agentic.agent.LLMManager import LLMManager
# groq, openai
llm = LLMManager.create_llm('ollama')
llm.call('Hello')

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
documents = loader.load()
print(f"Total documents {len(documents)}")
documents


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
_chunks = _splitter.split_documents(documents)
print(f"Total chunks {len(_chunks)}")
_chunks

In [ ]:
_chunk_text = [d.page_content for d in _chunks]
_chunk_text

In [ ]:
from langchain_ollama import OllamaEmbeddings
import numpy as np

# dimensions=512
embeddings = OllamaEmbeddings(model="nomic-embed-text")

_vectors = embeddings.embed_documents(_chunk_text)
my_array = np.array(_vectors)
print(f"embedding shape {my_array.shape}")
my_array

#### This example walks through building a retrieval augmented generation (RAG) application using Ollama and embedding models.

In [ ]:
# Step 1: Generate embeddings
import os
import chromadb

## Create a simple txt file
work_dir = os.getenv("WORK_DIR")
persist_directory = f"{work_dir}/vectorstore/chroma"
client = chromadb.PersistentClient(path=persist_directory)

#
collection = client.get_or_create_collection(name="docs")

In [ ]:
# Step 1: Generate embeddings
import ollama

documents = [
  "Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels",
  "Llamas were first domesticated and used as pack animals 4,000 to 5,000 years ago in the Peruvian highlands",
  "Llamas can grow as much as 6 feet tall though the average llama between 5 feet 6 inches and 5 feet 9 inches tall",
  "Llamas weigh between 280 and 450 pounds and can carry 25 to 30 percent of their body weight",
  "Llamas are vegetarians and have very efficient digestive systems",
  "Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old",
]

#client = chromadb.Client()
#collection = client.create_collection(name="docs")

# store each document in a vector embedding database
for i, d in enumerate(documents):
  # mxbai-embed-large
  response = ollama.embed(model="nomic-embed-text", input=d)
  embeddings = response["embeddings"]
  collection.add(
    ids=[str(i)],
    embeddings=embeddings,
    documents=[d]
  )

In [ ]:
# Step 2: Retrieve : the most relevant document given an example prompt:

import ollama

prompt = "What animals are llamas related to ?"

# generate an embedding for the input and retrieve the most relevant doc
response = ollama.embed(
  model="all-minilm",
  input=prompt
)

results = collection.query(
  query_embeddings=response["embeddings"],
  n_results=1
)

data = results['documents'][0][0]

In [ ]:
data

In [ ]:
#
# Step 3: Generate : response combining the prompt and data we retrieved in step 2
#
input = "What animals are llamas related to ?"
output = ollama.generate(
  model="llama3.2:1b-instruct-q8_0",
  prompt=f"Using this data: {data}. Respond to this prompt: {input}"
)

print(output['response'])

In [18]:
from crewai_tools import WebsiteSearchTool
from crewai import Agent, LLM
from crewai.project import agent
from crewai_tools import RagTool
from crewai.tools import tool
from com.example.agentic.tools.config import _tool_config

#@tool
def website_search(http_url: str):
    """
    Useful for when you need to ask with lookup on website data.
    """
    #return retriever.get_relevant_documents()
    # Proper Configuration Structure
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=_tool_config
    )

    return search_tool.run(http_url)

website_search("https://docs.crewai.com/how-to/Installing-CrewAI/")   

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping URL validation for: https://example.com
CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping URL validation for: https://example.com


'Relevant Content:\n\nreduced production incidents through proactive defect detection. \n\n  \n\n• Developed the physical data model for Investment Product & Portfolio Governance data pipelines considering \n\ndifferent functional & operational risk reporting. Drive the modernization of IPGC & UPGC Data Platform by \n\nmoving to Cloud Foundry/AWS & Oracle Cloud Services \n\n \n\n• Led the strategic modernization of IPGC & UPGC Data Platform from on-premise to Cloud Foundry, AWS, and \n\nOracle Cloud Services, reducing infrastructure costs by an estimated 30–40% and improving platform scalability \n\nto support 3× data volume growth without linear headcount addition. \n\n \n\n• Established architecture governance practices through structured design review sessions with SMEs and cross-\n\nfunctional teams, standardizing data engineering patterns and reducing rework cycles by ~25%, accelerating \n\ntime-to-production for new pipeline features. \n\n \n\nSR. SOFTWARE ENGINEER | ACCENTURE SE

In [ ]:
import ollama

single = ollama.embed(
  model='all-minilm',
  input='The sky is blue because of Rayleigh scattering'
)
print(single['embeddings'][0])  # vector length
print(len(single['embeddings'][0]))  # vector length

In [ ]:
researcher = Agent(
    llm=llm,
    name="Researcher",
    role="Researches topics by searching the website data.",
    tools=[website_search],
    goal="Answer questions by retrieving relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant specializing in searching and retrieving information from a website. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)

researcher_boss = Agent(
    llm=llm,
    name="Researcher Boss",
    role="Challenges the researcher to bring out the best out of his findings",
    tools=[website_search],
    goal="Ask further questions to the researcher and validates the retrieved relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant boss and your job is to make sure the retrieved information is correct. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)


In [ ]:
from crewai import Task
research_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher,
    #async_execution=True
)

boss_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher_boss,
    #async_execution=True
)


In [ ]:
from crewai import Crew
# Create Crew
research_crew1 = Crew(
    agents=[researcher, researcher_boss],
    tasks=[research_task, boss_task],
    verbose=True,  # This will print logs to the console as the crew works
    planning=True,
)

# Job Context
job_crew_works = {
    'crewai_url': 'https://docs.crewai.com/how-to/Installing-CrewAI/',
    'personal_writeup': """Accomplished Researcher 
    with 18 years of experience, specializing in
    setting up CrewAI kind of agent based systems"""
}

async_result = await research_crew1.kickoff_async(inputs=job_crew_works)
print(async_result)

# Kickoff the Crew's Work
#research_crew1.kickoff(inputs=job_crew_works)
#print(result)

In [ ]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from com.example.agentic.tools.config import _tool_config, _embedding_model_openai
# Create a RAG tool with custom configuration

rag_tool = RagTool(config=_tool_config, summarize=True)

In [4]:
# Add a PDF file
rag_tool.add(data_type="file", path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf")

# Add a web page
#rag_tool.add(data_type="web_page", url="https://example.com")

# Add a YouTube video
#rag_tool.add(data_type="youtube_video", url="https://www.youtube.com/watch?v=VIDEO_ID")

# Add a directory of files
#rag_tool.add(data_type="directory", path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs")

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


In [15]:
rag_tool.run(query="what is total experience")

"Relevant Content:\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 critical IT and operational risk \n\nitems (ORI, MERS, SERA), delivering solutions within agreed timelines and achieving full closure with zero \n\nregulatory escalations — strengthening UBS's control framework for IB Markets technology. \n\n \n\nVICE PRESIDENT – DATA & CLOUD TRANSFORMATION | JPMORGAN SERVICES | MUMBAI | MAY 2009 \n\n– MAR 2022 \n\n• Spearheaded a 13-year as Engineering Lead, multi-phase enterprise data platform (Investment Product & \n\nPortfolio Governance) serving Product & Portfolio Governance 